In [6]:
import pandas as pd
import numpy as np
import h5py
import os
# import import_ipynb
from datetime import timedelta

import sys
sys.path.append('C:/Users/wg984/Wolfgang/repos/sleep_research_io')
sys.path.append('/home/wolfgang/repos/sleep_research_io')

from sleep_research_functions import index_to_datetime_sleepdata, load_sleep_data, write_to_hdf5_file #, format_bm_airgo_to_10Hz_icusleep_data

sys.path.append('C:/Users/wg984/Wolfgang/repos/Bedmaster-ICU-tools/code')
sys.path.append('/home/wolfgang/repos/Bedmaster-ICU-tools/code')
# from bmresearch_tools import BMR_load, get_metadata

from resample_BMR import remove_non_monotonic_data

In [2]:

# import pytz

# def get_metadata(filepath):
#     '''
#     :param filepath: filepath to BMR .h5 file.
#     :return: list of signals contained in .h5 file.
#     '''
#     ff = h5py.File(filepath, 'r')
#     signals = [x for x in ff.keys() if (('_dt' not in x) and ('_event' not in x) and ('_segment' not in x))]
#     return signals

# def BMR_load(filepath, signals=None, DateTimeSelectionTZ=None, hours_to_return = 1, loadEvents = 0):  # BMR... BedMasterResearch

#     if signals == None: raise Exception(
#         'No signals given, i.e. load full signal. not implemented yet (ToDO: consider efficency).')

#     # load BM4Research .h5 file:
#     ff = h5py.File(filepath, 'r')

#     data = {}  # we collect all signals in a dictionary 'data'
#     for signaltmp in signals:
#         df = pd.DataFrame()
#         df_len = ff[signaltmp].shape[0]
#         if DateTimeSelectionTZ is not None:
#             # get DateTimes and Posix Times of specified DateTimeSelection:
#             PosixSelection = DateTimeSelectionTZ.timestamp()  # posix time of datetime selection
#             fs = np.max([int(1 / np.mean(np.diff(ff[signaltmp + '_dt'][:20]))),1])

#             # load time-info elements every hours_to_return to get an estimate where in the potentially large file the ROI is.
#             # we do not want to load full signal (large) but only neighborhood of DateTimeSelection
#             estimate_posix = ff[signaltmp + '_dt'][::fs * 3600 * hours_to_return][:]
#             estimate_idx = np.argmin(np.abs(estimate_posix - PosixSelection))
#             # with this estimate idx, load the data at this ROI:
#             start_idx = int(fs * 3600 * np.max([hours_to_return * estimate_idx - hours_to_return,0]))
#             end_idx = int(fs * 3600 * np.min([hours_to_return * estimate_idx + hours_to_return, df_len]))
#             df['signal'] = ff[signaltmp][start_idx:end_idx]
#             df['posix'] = ff[signaltmp + '_dt'][start_idx:end_idx]
#             if loadEvents:
#                 df['event'] = ff[signaltmp + '_event'][start_idx:end_idx]
#             # df['segment'] = ff[signaltmp + '_segment'][start_idx:end_idx]
#             df['datetime'] = [datetime.fromtimestamp(unixT, pytz.timezone('America/New_York')).replace(tzinfo=None) for
#                               unixT in df['posix'].values]
#         else:  # load full signal:
#             df['signal'] = ff[signaltmp][:]
#             df['posix'] = ff[signaltmp + '_dt'][:]
#             df['datetime'] = [datetime.fromtimestamp(unixT, pytz.timezone('America/New_York')).replace(tzinfo=None) for
#                               unixT in df['posix'].values]
#             if loadEvents: df['event'] = ff[signaltmp + '_event'][:]
#             # df['segment'] = ff[signaltmp + 'segment'][:]

#         # insert to data dict
#         data[signaltmp] = df

#     return data


In [3]:

def format_bm_airgo_to_10Hz_icusleep_data(bm_data, airgo_data):
    '''
    Input: AirGo and bedmaster data, original sampling rates.
    Output: 10Hz merged data, :='research format'
    '''
    data_list = [bm_data[signal_tmp] for signal_tmp in bm_data.keys()]
    data_list.append(airgo_data)
    data = pd.concat(data_list, join='outer', axis=1, sort=True)
    data = data.resample("0.1S").mean()

    for signal_tmp in bm_data.keys():
        data[signal_tmp] = data[signal_tmp].interpolate(method='pchip', order=3, limit_area='inside',
                              limit=50)
        data[signal_tmp + '_event'] = data[signal_tmp + '_event'].interpolate(method='pchip', order=3, limit_area='inside',
                              limit=50)

        data.loc[np.isnan(data[signal_tmp + '_event']), signal_tmp + '_event'] = 99 # replace nan with 99 because data gets saved as int in hdf5, nan not supported.

    return data
# limit considerations: 0.5Hz bedmaster to 10 Hz: for each datapoint, 19 NaNs.
# datagaps less than 5 seconds will be interpolated, if larger than 5 seconds, it is NaN. i.e. limit 50.


def format_only_airgo_to_10Hz_icusleep_data(airgo_data):
    '''
    Input: AirGo only, original sampling rates 10Hz.
    Output: 10Hz resampled/interpolated, :='research format'
    '''
    airgo_data = airgo_data.resample("0.1S").mean()
    return airgo_data
# limit considerations: 0.5Hz bedmaster to 10 Hz: for each datapoint, 19 NaNs.
# datagaps less than 5 seconds will be interpolated, if larger than 5 seconds, it is NaN. i.e. limit 50.

def format_only_bm_to_10Hz_icusleep_data(bm_data):
    
    data_list = [bm_data[signal_tmp] for signal_tmp in bm_data.keys()]
    data = pd.concat(data_list, join='outer', axis=1, sort=True)
    data = data.resample("0.1S").mean()

    for signal_tmp in bm_data.keys():
        data[signal_tmp] = data[signal_tmp].interpolate(method='pchip', order=3, limit_area='inside',
                              limit=50)
        data[signal_tmp + '_event'] = data[signal_tmp + '_event'].interpolate(method='pchip', order=3, limit_area='inside',
                              limit=50)

        data.loc[np.isnan(data[signal_tmp + '_event']), signal_tmp + '_event'] = 99 # replace nan with 99 because data gets saved as int in hdf5, nan not supported.

    return data
    

In [8]:
# load table containing time alignment information (offset to be applied)
tablefile = 'C:/Users/wg984/Wolfgang/repos/ICU-Sleep/data/bedmaster_airgo_timealignment.csv'
# tablefile = '/home/wolfgang/repos/ICU-Sleep/data/bedmaster_airgo_timealignment.csv'

table = pd.read_csv(tablefile, sep=';')
table = table.dropna(subset=['TS_manual'])

FileNotFoundError: [Errno 2] File /home/wolfgang/repos/ICU-Sleep/data/bedmaster_airgo_timealignment.csv does not exist: '/home/wolfgang/repos/ICU-Sleep/data/bedmaster_airgo_timealignment.csv'

In [5]:
table

,study_id,airgo_available,bmr_available,code_successful,note,TS_manual without 30 second shift,TS_manual,TS_auto1,TS_auto2,TS_auto3,confidence,further_notes
0,1,True,TRUE,1.0,NaN,-73.0,-43.0,-43.0,-772.0,-117.0,NaN,NaN
1,2,True,TRUE,1.0,NaN,-22.0,8.0,115.0,-302.0,-696.0,low,for exact match but trends are visible.
2,3,True,TRUE,1.0,many signals missing,-6.0,24.0,24.0,-28.0,-516.0,NaN,NaN
3,4,True,TRUE,1.0,signals look weird,-44.0,-14.0,-14.0,24.0,-220.0,NaN,NaN
5,6,True,TRUE,0.0,NaN,-44.0,-14.0,-13.0,-52.0,35.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
184,185,NaN,NaN,NaN,NaN,-60.0,-30.0,NaN,NaN,NaN,low confidence,NaN
185,186,NaN,NaN,NaN,NaN,-30.0,0.0,416.0,-51.0,707.0,low confidence,NaN
186,187,NaN,NaN,NaN,NaN,-55.0,-25.0,-25.0,-3.0,-434.0,NaN,NaN
187,188,NaN,NaN,NaN,NaN,-59.0,-39.0,-29.0,-39.0,-53.0,NaN,NaN


In [6]:
table.shape

(133, 12)

In [7]:
# airgo features dir:
# airgo_features_dir = 'E:/Boston_MGH/ICU-Sleep/airgo_features/'
airgo_features_dir = 'M:/Projects/ICU_SLEEP_STUDY/data/data_analysis/AirGo/airgo_features/'
# bedmaster research dir
# bm_research_dir = 'E:/Boston_MGH/ICU-Sleep/bm_research/'
bm_research_dir = 'M:/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_studyID/'
output_dir = 'M:/Projects/ICU_SLEEP_STUDY/data/data_analysis/biosignals_10Hz_data/'
output_dir_daynight = 'M:/Projects/ICU_SLEEP_STUDY/data/data_analysis/biosignals_10Hz_data_daynight/'

In [8]:
# get night id information:
data_partitions_id = 'M:/Projects/ICU_SLEEP_STUDY/data/data_analysis/Study/data_partitions_id.csv'
data_partitions_id = pd.read_csv(data_partitions_id)
data_partitions_id

,study_id,date,day_night,cam_icu,airgo,bedmaster,category,id
0,1,2018-06-05,day,False,False,True,C,001_d01
1,1,2018-06-05,night,False,False,True,C,001_n01
2,1,2018-06-06,day,False,True,True,A,001_d02
3,1,2018-06-06,night,False,True,True,A,001_n02
4,1,2018-06-07,day,False,False,True,C,001_d03
...,...,...,...,...,...,...,...,...
1850,189,2019-11-13,day,False,True,True,A,189_d10
1851,189,2019-11-13,night,False,True,True,A,189_n10
1852,189,2019-11-14,day,False,True,True,A,189_d11
1853,189,2019-11-14,night,False,True,True,A,189_n11


In [11]:
def save_night_day_partitions(data, partitions_studyid, output_dir_daynight, study_id):

    for irow in range(partitions_studyid.shape[0]):

        # get start and end datetime for selected day_night:
        di = partitions_studyid.iloc[irow].copy()
        start = pd.to_datetime(di.date, infer_datetime_format=1)
        if di.day_night == 'day':
            start = start.replace(hour=8)
        elif di.day_night == 'night':
            start = start.replace(hour=20)
        end = start + timedelta(hours=12)

        # save the datetime-selected data with day_night_id name info:
        output_h5_path = f'icusleep_{di.id}.h5'
        output_h5_path = os.path.join(output_dir_daynight, output_h5_path)
        
        if data.loc[start:end].shape[0] == 0:
            continue
            
        hdr = {
            'study_id': np.int32(study_id),
            'MRN': np.int32(1),
            'fs': np.int32(10),
            'start_datetime': np.array(
                [data.loc[start:end].index[0].year, data.loc[start:end].index[0].month, start.day,
                 data.loc[start:end].index[0].hour, data.loc[start:end].index[0].minute,
                 data.loc[start:end].index[0].second, data.loc[start:end].index[0].microsecond]),
            'day_night_id': di.id}

        # do the day/night selection
        data_sel = data.loc[start:end]
        
        for signal in data.columns:
            if data_sel[signal].dropna().shape[0] == 0:
                data_sel.drop(signal, axis=1, inplace=True)
        
        write_to_hdf5_file(data_sel, output_h5_path, hdr=hdr)

In [12]:
for study_id in range(1,190): # [76]: # [76, 155, 158, 185]: # range(1, 190):

#     if study_id == 27: continue # for now
        
    print(study_id)
    try:

        if os.path.exists(os.path.join(output_dir, 'icusleep_' + str(study_id).zfill(3) + '.h5')):
            print('already exists. continue.')
            continue

        if table.loc[table.study_id == study_id, 'TS_manual'].shape[0] > 0:
            # this usually means both AirGo and Bedmaster are available for this patient

            time_offset_to_be_applied = int(
                table.loc[table.study_id == study_id, 'TS_manual'].values[0])  # -60*60*24*3 # in seconds

            # load AirGo
            airgo_data = pd.read_csv(os.path.join(airgo_features_dir, 'airgo_' + str(study_id).zfill(3)) + '.csv')
            airgo_data.datetime = pd.to_datetime(airgo_data.datetime, infer_datetime_format=1)
            airgo_data.set_index('datetime', inplace=True)
            airgo_data.drop('crcstatus', axis=1, inplace=True)

            # load bedmaster data
            bmr_file = os.path.join(bm_research_dir, 'BMR_' + str(study_id).zfill(3) + '.h5')
            signals_in_bm_file = get_metadata(bmr_file)
            signals_to_load = ['NBPD', 'NBPS', 'HR', 'SPO2%', 'ART1S', 'ART1D', 'CUFF']
            signals_to_load = [x for x in signals_to_load if x in signals_in_bm_file]
            bm_data = BMR_load(bmr_file, signals=signals_to_load, loadEvents=1)

            bm_data = remove_non_monotonic_data(bm_data)

            for signal_tmp in signals_to_load:
                bm_data[signal_tmp].set_index('datetime', inplace=True)
                bm_data[signal_tmp].drop('posix', inplace=True, axis=1)
                bm_data[signal_tmp].rename(columns={'signal': signal_tmp}, inplace=True)
                bm_data[signal_tmp].rename(columns={'event': signal_tmp + '_event'}, inplace=True)

            #     # select only data from min(enrollment_day, first_cam_day) until last_cam_day+1 midnight.
            #     datetime_start =c
            #     datetime_end =
            #     bm_data[signal_tmp] = bm_data[signal_tmp].loc[datetime_start : datetime_end,:]

            # remove duplicate indices if still occuring:
            for signal in bm_data.keys():
                num_of_duplicates = bm_data[signal].index.duplicated().sum()
                bm_data[signal] = bm_data[signal].loc[~bm_data[signal].index.duplicated(keep='first')]

                if num_of_duplicates > 0:
                    print(f' bedmaster data still contains duplicated indices before resampling. simply drop them.'
                          f' # of duplicates = {num_of_duplicates}.')


            if 1:  # correct AirGo time so that it aligns with bedmaster
                airgo_data.index = airgo_data.index - timedelta(seconds=time_offset_to_be_applied)

                # somehow index can still have duplicates. if just a small number, drop the indices.
                if airgo_data.index.duplicated().sum() < 10:
                    airgo_data = airgo_data.loc[~airgo_data.index.duplicated(keep='first')]

                else:
                    raise ValueError('>10 duplicates in AirGo signal. To check.')

            data = format_bm_airgo_to_10Hz_icusleep_data(bm_data, airgo_data)

            hdr = {
                'study_id': np.int32(study_id),
                'MRN': np.int32(1),
                'fs': np.int32(10),
                'start_datetime': np.array(
                    [data.index[0].year, data.index[0].month, data.index[0].day, data.index[0].hour, data.index[0].minute,
                     data.index[0].second, data.index[0].microsecond])
            }

            output_h5_path = os.path.join(output_dir, 'icusleep_' + str(study_id).zfill(3) + '.h5')
            # save full data of study_id:
            write_to_hdf5_file(data, output_h5_path, hdr=hdr)

            # save night/day partitions
            partitions_studyid = data_partitions_id.loc[data_partitions_id.study_id == study_id]
            # to be checked if all information is passed to function:
            save_night_day_partitions(data, partitions_studyid, output_dir_daynight, study_id)

        elif table.loc[table.study_id == study_id, 'TS_manual'].shape[0] == 0:
            # this usually means, either only bedmaster or AirGo is available.

            # only if AirGo is available, process the data:
            airgo_path = os.path.join(airgo_features_dir, 'airgo_' + str(study_id).zfill(3)) + '.csv'
            airgo_available = os.path.exists(airgo_path)

            bm_path =  os.path.join(bm_research_dir, 'BMR_' + str(study_id).zfill(3) + '.h5')
            bm_available = os.path.exists(bm_path)

            if airgo_available & bm_available:
                print('UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.')

            if airgo_available:
                continue # only for now because this was already run. 

                # load AirGo
                airgo_data = pd.read_csv(airgo_path)
                airgo_data.datetime = pd.to_datetime(airgo_data.datetime, infer_datetime_format=1)
                airgo_data.set_index('datetime', inplace=True)
                airgo_data.drop('crcstatus', axis=1, inplace=True)

                # somehow index can still have duplicates. if just a small number, drop the indices.
                if airgo_data.index.duplicated().sum() < 10:
                    airgo_data = airgo_data.loc[~airgo_data.index.duplicated(keep='first')]

                else:
                    raise ValueError('>10 duplicates in AirGo signal. To check.')


                data = format_only_airgo_to_10Hz_icusleep_data(airgo_data)


            elif bm_available:

                # load bedmaster data
                bmr_file = os.path.join(bm_research_dir, 'BMR_' + str(study_id).zfill(3) + '.h5')
                signals_in_bm_file = get_metadata(bmr_file)
                signals_to_load = ['NBPD', 'NBPS', 'HR', 'SPO2%', 'ART1S', 'ART1D', 'CUFF']
                signals_to_load = [x for x in signals_to_load if x in signals_in_bm_file]
                bm_data = BMR_load(bmr_file, signals=signals_to_load, loadEvents=1)

                bm_data = remove_non_monotonic_data(bm_data)

                for signal_tmp in signals_to_load:
                    bm_data[signal_tmp].set_index('datetime', inplace=True)
                    bm_data[signal_tmp].drop('posix', inplace=True, axis=1)
                    bm_data[signal_tmp].rename(columns={'signal': signal_tmp}, inplace=True)
                    bm_data[signal_tmp].rename(columns={'event': signal_tmp + '_event'}, inplace=True)

                # remove duplicate indices if still occuring:
                for signal in bm_data.keys():
                    num_of_duplicates = bm_data[signal].index.duplicated().sum()
                    bm_data[signal] = bm_data[signal].loc[~bm_data[signal].index.duplicated(keep='first')]

                    if num_of_duplicates > 0:
                        print(f' bedmaster data still contains duplicated indices before resampling. simply drop them.'
                              f' # of duplicates = {num_of_duplicates}.')
                    # somehow index can still have duplicates. if just a small number, drop the indices.
                    if airgo_data.index.duplicated().sum() < 10:
                        airgo_data = airgo_data.loc[~airgo_data.index.duplicated(keep='first')]
                    else:
                        raise ValueError('>10 duplicates in AirGo signal. To check.')

                data = format_only_bm_to_10Hz_icusleep_data(bm_data)

            else: # no data available at all.
                continue

            # and save the data:    
            hdr = {
                'study_id': np.int32(study_id),
                'MRN': np.int32(1),
                'fs': np.int32(10),
                'start_datetime': np.array(
                    [data.index[0].year, data.index[0].month, data.index[0].day, data.index[0].hour, data.index[0].minute,
                     data.index[0].second, data.index[0].microsecond])
            }

            output_h5_path = os.path.join(output_dir, 'icusleep_' + str(study_id).zfill(3) + '.h5')
            # save full data of study_id:
            write_to_hdf5_file(data, output_h5_path, hdr=hdr)

            # save night/day partitions
            partitions_studyid = data_partitions_id.loc[data_partitions_id.study_id == study_id]
            # to be checked if all information is passed to function:
            save_night_day_partitions(data, partitions_studyid, output_dir_daynight, study_id)


    except Exception as e:
        print(e)

1


C:\Users\wg984\AppData\Local\Continuum\anaconda3\lib\site-packages\pandas\core\frame.py:4117: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



2
3
4
5
6
7
8
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
9
10
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
11
12
13
14
15
16
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
17
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
18
19
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
20
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
21
22
23
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
24
25
26
27
UNEXPECTED. no point_a_idx in removing_non_monotonicity.

28
29
30
31
32
33
34
35
36
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
37
38
39
40
41
42
43
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61


C:\Users\wg984\AppData\Local\Continuum\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3063: DtypeWarning:

Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.



62
63
64
65
66
67
68
69
70
71
72
73
74
75
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
128
129
130
131
132
133
134
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
135
136
137
138
139
140
141
142
143
144
145
146
UNEXPECTED. BOTH FILES EXIST BUT APPARENTLY DO NOT HAVE A VALUE IN TABLE FILE.
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
 bedmaster data still contains duplicated indices before resampling. simply drop them. # of duplicates = 1779.
 bedmaster data still contains duplicated indices before resampling. simply drop them. # of dupl

In [ ]:
# even though data_availability and night_day_categorization scripts have been written and executed before this script, let's do something very similar here.
# difference: input: data that we REALLY have, i.e. all processed and in the research format. i.e. extract metadata from there.

In [26]:
datadir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/biosignals_10Hz_data'

In [27]:
files = os.listdir(datadir)

In [28]:
len(files)

130

In [12]:
pd.options.display.max_rows = 99

In [27]:
i = 1500
data.iloc[i:i+100][['HR','band','SPO2%']]

,HR,band,SPO2%
datetime,,,
2019-04-01 17:43:47.000,100.0,NaN,100.0
2019-04-01 17:43:47.100,100.0,NaN,100.0
2019-04-01 17:43:47.200,100.0,NaN,100.0
2019-04-01 17:43:47.300,100.0,NaN,100.0
2019-04-01 17:43:47.400,100.0,NaN,100.0
...,...,...,...
2019-04-01 17:43:56.500,100.0,NaN,100.0
2019-04-01 17:43:56.600,100.0,NaN,100.0
2019-04-01 17:43:56.700,100.0,NaN,100.0


### 

,study_id,date,day_night,cam_icu,airgo,bedmaster,category,id
0,1,2018-06-05,day,False,False,True,C,001_d01
1,1,2018-06-05,night,False,False,True,C,001_n01
2,1,2018-06-06,day,False,True,True,A,001_d02
3,1,2018-06-06,night,False,True,True,A,001_n02
4,1,2018-06-07,day,False,False,True,C,001_d03
...,...,...,...,...,...,...,...,...
1795,189,2019-11-13,day,False,True,True,A,189_d10
1796,189,2019-11-13,night,False,True,True,A,189_n10
1797,189,2019-11-14,day,False,True,True,A,189_d11
1798,189,2019-11-14,night,False,True,True,A,189_n11


In [14]:
partitions_studyid

,study_id,date,day_night,cam_icu,airgo,bedmaster,category,id
343,36,2019-02-04,day,False,True,False,B,036_d01
344,36,2019-02-04,night,False,True,False,B,036_n01
345,36,2019-02-05,day,False,True,False,B,036_d02
